# Paper Trading Results

Analyze paper trading performance from `trades.json`. Fetches settlement results from Kalshi to compute P&L.

**Metrics:** Returns, Win %, Max Drawdown (daily), Sharpe Ratio (annualized)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime

import config
from kalshi_client import KalshiClient

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["figure.dpi"] = 120

## 1. Load Trades

In [ ]:
with open("trades.json", "r") as f:
    raw_trades = json.load(f)

trades = pd.DataFrame(raw_trades)
trades["timestamp"] = pd.to_datetime(trades["timestamp"])
trades["date"] = trades["timestamp"].dt.date

print(f"Loaded {len(trades)} trades from {trades['date'].min()} to {trades['date'].max()}")
print(f"Cities traded: {trades['city'].nunique()}")
print(f"Mode: {trades['mode'].unique()}")
trades.head(10)

## 2. Fetch Settlement Results

In [ ]:
client = KalshiClient(config.KALSHI_API_KEY_ID, config.KALSHI_PRIVATE_KEY_PATH)

# Fetch settlement result for each unique ticker
tickers = trades["ticker"].unique()
print(f"Fetching settlement data for {len(tickers)} contracts...")

settlements = {}
for i, ticker in enumerate(tickers):
    try:
        mkt = client.get_market(ticker)
        settlements[ticker] = {
            "status": mkt.get("status"),
            "result": mkt.get("result"),           # "yes" or "no"
            "title": mkt.get("title", ""),
            "floor_strike": mkt.get("floor_strike"),
            "cap_strike": mkt.get("cap_strike"),
        }
    except Exception as e:
        print(f"  Error fetching {ticker}: {e}")
        settlements[ticker] = {"status": "error", "result": None}

settle_df = pd.DataFrame.from_dict(settlements, orient="index")
settle_df.index.name = "ticker"

settled = settle_df[settle_df["result"].notna()]
unsettled = settle_df[settle_df["result"].isna()]
print(f"\nSettled: {len(settled)}  |  Unsettled/open: {len(unsettled)}")
settle_df.head()

## 3. Compute P&L

In [ ]:
# Merge settlement results into trades
trades = trades.merge(
    settle_df[["result", "status"]],
    left_on="ticker",
    right_index=True,
    how="left",
)

# Filter to settled trades only for P&L
settled_trades = trades[trades["result"].notna()].copy()
print(f"Settled trades: {len(settled_trades)} / {len(trades)}")

# P&L per trade (all YES side)
# Win (result="yes"): payout=100, P&L = (100 - entry_price - fee) * contracts
# Loss (result="no"): payout=0,  P&L = (0 - entry_price - fee) * contracts
settled_trades["payout_cents"] = settled_trades["result"].map({"yes": 100, "no": 0})
settled_trades["pnl_per_contract"] = (
    settled_trades["payout_cents"] - settled_trades["entry_price_cents"] - settled_trades["fee_cents"]
)
settled_trades["pnl_cents"] = settled_trades["pnl_per_contract"] * settled_trades["contracts"]
settled_trades["pnl_dollars"] = settled_trades["pnl_cents"] / 100
settled_trades["cost_cents"] = (settled_trades["entry_price_cents"] + settled_trades["fee_cents"]) * settled_trades["contracts"]
settled_trades["cost_dollars"] = settled_trades["cost_cents"] / 100
settled_trades["won"] = settled_trades["result"] == "yes"

settled_trades[["date", "city", "ticker", "entry_price_cents", "fee_cents",
                "payout_cents", "pnl_per_contract", "pnl_dollars", "won"]].head(10)

## 4. Summary Statistics

In [ ]:
total_trades = len(settled_trades)
wins = settled_trades["won"].sum()
losses = total_trades - wins
win_pct = wins / total_trades * 100 if total_trades else 0

total_pnl = settled_trades["pnl_dollars"].sum()
total_cost = settled_trades["cost_dollars"].sum()
total_return_pct = (total_pnl / total_cost * 100) if total_cost else 0

avg_win = settled_trades.loc[settled_trades["won"], "pnl_dollars"].mean() if wins else 0
avg_loss = settled_trades.loc[~settled_trades["won"], "pnl_dollars"].mean() if losses else 0

# Daily P&L for drawdown and Sharpe
daily_pnl = settled_trades.groupby("date")["pnl_dollars"].sum()

# Max drawdown (daily cumulative)
cumulative = daily_pnl.cumsum()
running_max = cumulative.cummax()
drawdown = cumulative - running_max
max_drawdown = drawdown.min()

# Sharpe ratio (annualized, assuming 365 trading days for event markets)
if len(daily_pnl) > 1 and daily_pnl.std() > 0:
    sharpe = (daily_pnl.mean() / daily_pnl.std()) * np.sqrt(365)
else:
    sharpe = 0.0

print("=" * 50)
print("       PAPER TRADING RESULTS")
print("=" * 50)
print(f"  Period:          {settled_trades['date'].min()} → {settled_trades['date'].max()}")
print(f"  Total trades:    {total_trades}")
print(f"  Wins / Losses:   {wins} / {losses}")
print(f"  Win rate:        {win_pct:.1f}%")
print(f"  Avg win:         ${avg_win:,.2f}")
print(f"  Avg loss:        ${avg_loss:,.2f}")
print(f"──────────────────────────────────────────────────")
print(f"  Total P&L:       ${total_pnl:,.2f}")
print(f"  Total deployed:  ${total_cost:,.2f}")
print(f"  Return:          {total_return_pct:+.2f}%")
print(f"  Max drawdown:    ${max_drawdown:,.2f}")
print(f"  Sharpe (ann.):   {sharpe:.2f}")
print("=" * 50)

## 5. Cumulative P&L

In [ ]:
fig, ax = plt.subplots()

dates = pd.to_datetime(pd.Series(cumulative.index))
ax.plot(dates, cumulative.values, linewidth=2, color="#2196F3")
ax.fill_between(dates, cumulative.values, alpha=0.15, color="#2196F3")
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")

ax.set_title("Cumulative P&L (Daily)", fontsize=14, fontweight="bold")
ax.set_ylabel("P&L ($)")
ax.set_xlabel("Date")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Daily P&L Bar Chart

In [ ]:
fig, ax = plt.subplots()

dates = pd.to_datetime(pd.Series(daily_pnl.index))
colors = ["#4CAF50" if v >= 0 else "#F44336" for v in daily_pnl.values]

ax.bar(dates, daily_pnl.values, color=colors, width=0.8)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")

ax.set_title("Daily P&L", fontsize=14, fontweight="bold")
ax.set_ylabel("P&L ($)")
ax.set_xlabel("Date")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 7. Drawdown

In [ ]:
fig, ax = plt.subplots()

dates = pd.to_datetime(pd.Series(drawdown.index))
ax.fill_between(dates, drawdown.values, alpha=0.4, color="#F44336")
ax.plot(dates, drawdown.values, color="#F44336", linewidth=1.5)
ax.axhline(0, color="gray", linewidth=0.8, linestyle="--")

# Mark max drawdown point
max_dd_date = pd.to_datetime(drawdown.idxmin())
ax.annotate(
    f"Max DD: ${max_drawdown:,.2f}",
    xy=(max_dd_date, max_drawdown),
    xytext=(max_dd_date, max_drawdown * 0.6),
    arrowprops=dict(arrowstyle="->", color="#D32F2F"),
    fontsize=11, fontweight="bold", color="#D32F2F",
)

ax.set_title("Drawdown (Daily)", fontsize=14, fontweight="bold")
ax.set_ylabel("Drawdown ($)")
ax.set_xlabel("Date")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 8. Win Rate by City

In [ ]:
city_stats = settled_trades.groupby("city").agg(
    trades=("won", "count"),
    wins=("won", "sum"),
    pnl=("pnl_dollars", "sum"),
).sort_values("pnl", ascending=True)

city_stats["win_pct"] = (city_stats["wins"] / city_stats["trades"] * 100).round(1)

fig, axes = plt.subplots(1, 2, figsize=(16, max(5, len(city_stats) * 0.4)))

# P&L by city
colors = ["#4CAF50" if v >= 0 else "#F44336" for v in city_stats["pnl"]]
axes[0].barh(city_stats.index, city_stats["pnl"], color=colors)
axes[0].axvline(0, color="gray", linewidth=0.8, linestyle="--")
axes[0].set_title("P&L by City ($)", fontsize=13, fontweight="bold")
axes[0].set_xlabel("P&L ($)")

# Win rate by city
axes[1].barh(city_stats.index, city_stats["win_pct"], color="#2196F3")
axes[1].axvline(50, color="gray", linewidth=0.8, linestyle="--")
axes[1].set_title("Win Rate by City (%)", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Win %")
axes[1].set_xlim(0, 100)

plt.tight_layout()
plt.show()

city_stats.sort_values("pnl", ascending=False)

## 9. Entry Price Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

win_prices = settled_trades.loc[settled_trades["won"], "entry_price_cents"]
loss_prices = settled_trades.loc[~settled_trades["won"], "entry_price_cents"]

bins = range(91, 98)
ax.hist(win_prices, bins=bins, alpha=0.7, label=f"Wins ({len(win_prices)})", color="#4CAF50", edgecolor="white")
ax.hist(loss_prices, bins=bins, alpha=0.7, label=f"Losses ({len(loss_prices)})", color="#F44336", edgecolor="white")

ax.set_title("Entry Price Distribution (Wins vs Losses)", fontsize=14, fontweight="bold")
ax.set_xlabel("Entry Price (cents)")
ax.set_ylabel("Count")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Unsettled Positions

Trades still open / not yet settled.

In [ ]:
open_trades = trades[trades["result"].isna()]

if open_trades.empty:
    print("All trades are settled.")
else:
    print(f"{len(open_trades)} trades still open / unsettled:")
    open_cost = (open_trades["entry_price_cents"] + open_trades["fee_cents"]) * open_trades["contracts"]
    print(f"Capital at risk: ${open_cost.sum() / 100:,.2f}")
    open_trades[["date", "city", "ticker", "contracts", "entry_price_cents", "fee_cents"]]